# Tree-Based Modeling

## Objective

The previous notebook showed that simple linear models struggled to model the dataset effectively.

Even after applying:

- Ridge regularization
- log-target transformation
- cross-validation
- RMSLE evaluation

performance remained limited overall.

This suggests that the dataset contains important non-linear relationships that cannot be captured well by linear regression models.

The goal of this notebook is to investigate whether tree-based models can better learn the underlying structure of the data.

---

# Why Tree-Based Models?

Linear models assume mostly linear relationships between features and the target variable.

However, many real-world datasets contain:

- non-linear interactions
- threshold effects
- feature combinations
- local decision patterns

Tree-based models can capture these patterns automatically.

Unlike linear regression, tree models:

- split observations into regions
- learn hierarchical decision rules
- model interactions without manual specification
- handle high-dimensional numerical data effectively

This makes them strong candidates for the current dataset.

---

# Modeling Strategy

This notebook will follow the progression below:

1. Rebuild the Random Forest baseline
2. Train a single Decision Tree model
3. Compare Decision Tree vs Random Forest
4. Evaluate fold stability across models
5. Add engineered row-level features
6. Re-evaluate Random Forest performance
7. Tune Random Forest hyperparameters
8. Analyze feature importance
9. Summarize findings and modeling direction

---

# Expected Outcome

By the end of this notebook, we should understand:

- whether tree-based models outperform linear models
- whether ensemble methods improve generalization
- whether engineered features improve predictive performance
- which hyperparameters influence performance most
- whether the model shows signs of overfitting
- which modeling direction should be pursued next

## 1. Imports

In [2]:
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import (
    KFold,
    cross_validate,
    GridSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## 2. Reproducibility Setup

In [3]:
RANDOM_STATE = 42

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

## 3. Load Prepared Dataset

In [7]:
train = pd.read_csv("./data/train.csv")
test = pd.read_csv("./data/test.csv")

y = train["target"]
X = train.drop(columns=["ID", "target"])
X_test = test.drop(columns=["ID"])

## 4. Remove Constant Features

In [8]:
feature_variances = X.var()

constant_cols = feature_variances[
    feature_variances == 0
].index

X = X.drop(columns=constant_cols)
X_test = X_test.drop(columns=constant_cols)

print("Removed constant features:", len(constant_cols))
print("X shape:", X.shape)
print("X_test shape:", X_test.shape)

Removed constant features: 256
X shape: (4459, 4735)
X_test shape: (49342, 4735)


## 5. Decision Tree Baseline

In [11]:
tree_model = DecisionTreeRegressor(
    random_state=RANDOM_STATE
)

tree_scores = cross_validate(
    tree_model,
    X,
    y,
    cv=kf,
    scoring=(
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error",
        "r2"
    ),
    return_train_score=True
)

In [12]:
tree_mae = -tree_scores["test_neg_mean_absolute_error"]
tree_rmse = -tree_scores["test_neg_root_mean_squared_error"]
tree_r2 = tree_scores["test_r2"]

print("Decision Tree Results")
print("MAE:", tree_mae.mean())
print("RMSE:", tree_rmse.mean())
print("R²:", tree_r2.mean())

Decision Tree Results
MAE: 6134082.857312729
RMSE: 9880458.85671768
R²: -0.46135002918489204


## 6. Random Forest Baseline


In [15]:
forest_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_estimators=50,
    max_depth=20,
    n_jobs=-1
)

forest_scores = cross_validate(
    forest_model,
    X,
    y,
    cv=kf,
    scoring=(
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error",
        "r2"
    ),
    return_train_score=True
)

forest_mae = -forest_scores["test_neg_mean_absolute_error"]
forest_rmse = -forest_scores["test_neg_root_mean_squared_error"]
forest_r2 = forest_scores["test_r2"]

print("Random Forest Results")
print("MAE:", forest_mae.mean())
print("RMSE:", forest_rmse.mean())
print("R²:", forest_r2.mean())

Random Forest Results
MAE: 5031415.062305139
RMSE: 7262026.002143969
R²: 0.21644454014504796


## 7. Cross-Validation Stability Analysis

In [16]:
print("Decision Tree R² Std:", np.std(tree_r2))
print("Random Forest R² Std:", np.std(forest_r2))

Decision Tree R² Std: 0.1573808482473252
Random Forest R² Std: 0.03124436946599884


# Baseline Conclusion

The baseline experiments showed major differences between linear and tree-based approaches.

Linear regression and Ridge Regression performed very poorly, with strongly negative R² scores. This suggests that the relationship between the features and the target variable is highly non-linear and cannot be captured effectively by linear models.

A single Decision Tree improved performance slightly but remained unstable across validation folds, indicating high sensitivity to the training data.

Random Forest produced the strongest baseline performance:

- lower MAE
- lower RMSE
- positive R² scores
- improved fold stability

This suggests that ensemble tree methods are better suited for the dataset because they can capture complex non-linear interactions while reducing variance compared to a single tree.

The cross-validation variance analysis also showed that Random Forest generalized more consistently across folds than the Decision Tree baseline.

These findings establish tree-based ensemble models as the most promising direction for further model improvement and hyperparameter tuning.

# Final Conclusion

This notebook tested tree-based baseline models after the linear baseline models from the previous notebook performed poorly.

The Decision Tree improved compared to linear regression but still showed weak and unstable validation performance.

The Random Forest baseline performed best overall, achieving a positive R² score and lower error values than the single Decision Tree.

This suggests that ensemble tree models are a promising direction because they can capture non-linear structure while reducing the instability of individual trees.

Further improvements should not be added in this notebook. They will be handled in later project stages:

- feature engineering
- feature validation
- hyperparameter tuning
- model expansion